In [1]:
# Libraries

# For Python to communicate with UTD's calendar API automatically
import requests

# Convert into organised table
import pandas as pd

# Introduces short delay between API requests
import time

In [2]:
# Linking to UTD calendar

url = "https://calendar.utdallas.edu/api/2/events"

params = {
    "days": 30, # Obtains events in next 30 days
    "pp": 10, # Obtains max 10 events per page
    "page": 1 # Start on 1st page
}

response = requests.get( # Sends request to UTD
    url,
    params=params,
    timeout=20 # If no response in 20 seconds from server then goes into timeout
)

response.raise_for_status() # Produces error with HTTP responses that are unsuccessful

print("Status:", response.status_code) 

Status: 200


In [3]:
# JSON conversion

data = response.json() # JSON changes to Python data structures

print (type(data)) # Type of python object received
print(data.keys()) # Which fields are accessible 

<class 'dict'>
dict_keys(['events', 'page', 'date'])


In [4]:
# Obtain the events

events = data["events"] # Retrieves event list

print("Events obtained:", len(events))

first_event = events[0]["event"] # First item

print("First event:", first_event["title"]) # Event name

Events obtained: 10
First event: Apply for your U.S. Passport


In [5]:
# JSON into Pandas 

# Loops through each item and retrieves inner information
event_records = [
    item["event"] for item in events
]

df = pd.json_normalize(event_records) # Converts JSON information into table

print(df.shape)

df.head()

(10, 72)


,id,title,url,updated_at,created_at,facebook_id,first_date,last_date,hashtag,urlname,...,geo.country,geo.zip,custom_fields.contact_information_name,custom_fields.contact_information_email,custom_fields.contact_information_phone,venue_url,filters.event_target_audience,filters.event_topic,filters.event_types,custom_fields.parking
0,54382811795627,Apply for your U.S. Passport,https://oics.utdallas.edu/passport-services/,2026-11-17T05:43:24-06:00,2026-10-30T13:49:06-05:00,None,2026-11-16,2026-11-20,IEW,apply-for-your-us-passport,...,US,75252,UTD Passport Services,passport@utdallas.edu,972-883-4715,NaN,NaN,NaN,NaN,NaN
1,54310776832134,Celebrating International Cinema,NaN,2026-11-17T05:41:13-06:00,2026-10-22T10:24:40-05:00,None,2026-11-16,2026-11-20,IEW,celebrating-international-cinema,...,US,75080,Library Research and Instruction,libinstructionservices@utdallas.edu,972-883-2847,https://calendar.utdallas.edu/mc,"[{'name': 'Undergraduate Students', 'id': 3064...","[{'name': 'Arts & Culture', 'id': 306451845423...","[{'name': 'Arts & Performances', 'id': 3064518...",NaN
2,49277555244922,"Fall 2026 ""Signature Required"" Period to Withd...",NaN,2026-11-17T04:35:46-06:00,2025-04-01T12:55:25-05:00,None,2026-10-27,2026-11-25,NaN,fall-2026-signature-required-period-to-withdra...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[{'name': 'Academic Calendar', 'id': 306451845...",NaN,NaN
3,53319204232423,Fall 2026 Final day to upload final version of...,NaN,2026-11-16T06:46:07-06:00,2026-07-02T08:35:53-05:00,None,2026-11-17,2026-11-17,NaN,fall-2026-final-day-to-upload-final-version-of...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[{'name': 'Academic Calendar', 'id': 306451845...",NaN,NaN
4,54036739156426,Hunger and Homelessness Awareness Week,https://basicneeds.utdallas.edu/events/hhweek/,2026-11-17T05:33:02-06:00,2026-09-21T11:02:10-05:00,None,2026-11-16,2026-11-20,NaN,hunger-and-homelessness-awareness-week-2026,...,NaN,NaN,Basic Needs Resource Center,basicneeds@utdallas.edu,972-883-6515,NaN,"[{'name': 'Undergraduate Students', 'id': 3064...","[{'name': 'Health & Wellness', 'id': 306451845...","[{'name': 'Campus & Community', 'id': 30645180...",NaN


Now that I have established a link to the UTD Comet Calendar API and can obtain event data, I can now make an automatic data collection process via pagination.

In [6]:
# Automate Pagination

every_event = [] # Makes empty list to store records of event

for page in range(1,6): # Loops pages 1 to 5

    params = {
        "days": 30,
        "pp": 100,
        "page": page
    }

    response = requests.get(
        url,
        params = params,
        timeout = 20
    )

    response.raise_for_status()
    data = response.json()

    events = data.get("events", []) # Obtains events

    if len(events) == 0: #If page contains no events, then stops loop
        break

    for item in events: 
        every_event.append(item["event"]) # Adds item to list 

    print(f"Page {page}: {len(events)} events") # Variable printed

    if len(events) < 100:
        break

    time.sleep(1)

df = pd.json_normalize(every_event)

print("Total records:", len(df))


Page 1: 100 events
Page 2: 100 events
Page 3: 100 events
Page 4: 100 events
Page 5: 42 events
Total records: 442


In [7]:
# Dataset exploration

df.shape

df.columns.tolist()

df.head()

df.isnull().sum().sort_values(ascending=False)

facebook_id              442
campus_id                442
conference_event_id      442
custom_fields.excerpt    437
groups                   435
                        ... 
keywords                   0
tags                       0
description                0
description_text           0
id                         0
Length: 74, dtype: int64

In [8]:
# Choosing columns to use for initial simplified dataset

columns = [
    "id", 
    "title",
    "description_text",
    "first_date", 
    "last_date",
    "location_name",
    "tags",
    "localist_url",
]

df_events = df[columns].copy()

df_events = df_events.drop_duplicates(
    subset = "id"
)

df_events.head()

,id,title,description_text,first_date,last_date,location_name,tags,localist_url
0,54382811795627,Apply for your U.S. Passport,"During International Education Week, all photo...",2026-11-16,2026-11-20,ROC,"[iew, oics]",https://calendar.utdallas.edu/event/apply-for-...
1,54310776832134,Celebrating International Cinema,To celebrate International Education Week (IEW...,2026-11-16,2026-11-20,Eugene McDermott Library (MC),[iew],https://calendar.utdallas.edu/event/celebratin...
2,49277555244922,"Fall 2026 ""Signature Required"" Period to Withd...",,2026-10-27,2026-11-25,,[],https://calendar.utdallas.edu/event/fall-2026-...
3,53319204232423,Fall 2026 Final day to upload final version of...,,2026-11-17,2026-11-17,,[],https://calendar.utdallas.edu/event/fall-2026-...
4,54036739156426,Hunger and Homelessness Awareness Week,"The Basic Needs Resource Center (BNRC), Office...",2026-11-16,2026-11-20,,"[basic needs, basic needs resource center]",https://calendar.utdallas.edu/event/hunger-and...


In [9]:
df_events.to_csv( 
    "utd_eventsData.csv",
    index = False #Stops Pandas adding its own row-number column to file
)

print("Dataset saved") 

Dataset saved
